# Exercise 10-2 — Music description with Essentia

In this exercise you extend the sound clustering from E9 to a larger and more challenging problem: 10 instrument classes instead of 3. You will gain hands-on experience with **Essentia** for feature extraction and develop intuition for why descriptor quality and selection matter as the number of classes grows.

In E9, clustering with 3 instruments was tractable with Freesound's pre-computed descriptors. Scaling to 10 classes makes the problem harder in two ways: more inter-class confusion is expected, and descriptor noise (e.g., from silence at the start of a recording) degrades performance more. This exercise guides you through a structured improvement pipeline:

1. **Part 1** — Download 10-class sound collections.
2. **Part 2** — Establish a baseline k-means clustering accuracy using Freesound descriptors.
3. **Part 3** — Improve accuracy by computing richer descriptors with Essentia and stripping silent frames.

**Essentia setup:**
- Download and install: http://essentia.upf.edu/
- Documentation: http://essentia.upf.edu/documentation/index.html
- If local installation is not feasible, run in the Docker image provided with sms-tools: `docker-compose up` from the root directory starts a Jupyter server with Essentia included.

In [ ]:
import os, sys
import numpy as np
import json
import freesound as fs
from scipy.cluster.vq import vq, kmeans, whiten

descriptors = [ 'lowlevel.spectral_centroid.mean',
                'lowlevel.spectral_contrast.mean',
                'lowlevel.dissonance.mean',
                'lowlevel.hfc.mean',
                'lowlevel.mfcc.mean',
                'sfx.logattacktime.mean',
                'sfx.inharmonicity.mean']

# Mapping of descriptors
descriptorMapping = { 0: 'lowlevel.spectral_centroid.mean',
                      1: 'lowlevel.dissonance.mean',
                      2: 'lowlevel.hfc.mean',
                      3: 'sfx.logattacktime.mean',
                      4: 'sfx.inharmonicity.mean',
                      5: 'lowlevel.spectral_contrast.mean.0',
                      6: 'lowlevel.spectral_contrast.mean.1',
                      7: 'lowlevel.spectral_contrast.mean.2',
                      8: 'lowlevel.spectral_contrast.mean.3',
                      9: 'lowlevel.spectral_contrast.mean.4',
                      10: 'lowlevel.spectral_contrast.mean.5',
                      11: 'lowlevel.mfcc.mean.0',
                      12: 'lowlevel.mfcc.mean.1',
                      13: 'lowlevel.mfcc.mean.2',
                      14: 'lowlevel.mfcc.mean.3',
                      15: 'lowlevel.mfcc.mean.4',
                      16: 'lowlevel.mfcc.mean.5'
                    }

## Part 1 – Download sound collections

Download at least 10 instrument classes from the following list: violin, guitar, bassoon, trumpet, clarinet, cello, naobo, snare drum, flute, mridangam, daluo, xiaoluo.

For each class, use `download_sounds_freesound()` to download 20 representative single-note or single-stroke examples. Use the **high-quality MP3 preview** (`preview_hq_mp3`) since these files will also be processed with Essentia.

- Call `download_sounds_freesound()` for each instrument. Design `queryText`, `tag`, and `duration` to maximise collection coherence.

In [ ]:
def download_sounds_freesound(queryText = "", tag=None, duration=None, API_Key = "", outputDir = "", topNResults = 5, featureExt = '.json'):
  """
  This function downloads sounds and their descriptors from freesound using the queryText and the 
  tag specified in the input. Additionally, you can also specify the duration range to filter sounds 
  based on duration.
  
  Inputs:
        (Input parameters marked with a * are optional)
        queryText (string): query text for the sounds (eg. "violin", "trumpet", "cello", "bassoon" etc.)
        tag* (string): tag to be used for filtering the searched sounds. (eg. "multisample",  
                       "single-note" etc.)
        duration* (tuple): min and the max duration (seconds) of the sound to filter, eg. (0.2,15)
        API_Key (string): your api key, which you can obtain from : www.freesound.org/apiv2/apply/
        outputDir (string): path to the directory where you want to store the sounds and their 
                            descriptors
        topNResults (integer): number of results(sounds) that you want to download 
        featureExt (string): file extension for storing sound descriptors
  output:
        This function downloads sounds and descriptors, and then stores them in outputDir. In 
        outputDir it creates a directory of the same name as that of the queryText. In this 
        directory outputDir/queryText it creates a directory for every sound with the name 
        of the directory as the sound id. Additionally, this function also dumps a text file 
        containing sound-ids and freesound links for all the downloaded sounds in the outputDir. 
        NOTE: If the directory outputDir/queryText exists, it deletes the existing contents 
        and stores only the sounds from the current query. 
  """ 
  
  # Checking for the compulsory input parameters
  if queryText == "":
    print("\n")
    print("Provide a query text to search for sounds")
    return -1
    
  if API_Key == "":
    print("\n")
    print("You need a valid freesound API key to be able to download sounds.")
    print("Please apply for one here: www.freesound.org/apiv2/apply/")
    print("\n")
    return -1
    
  if outputDir == "" or not os.path.exists(outputDir):
    print("\n")
    print("Please provide a valid output directory. This will be the root directory for storing sounds and descriptors")
    return -1    
  
  # Setting up the Freesound client and the authentication key
  fsClnt = fs.FreesoundClient()
  fsClnt.set_token(API_Key,"token")  
  
  # Creating a duration filter string that the Freesound API understands
  if duration and type(duration) == tuple:
    flt_dur = " duration:[" + str(duration[0])+ " TO " +str(duration[1]) + "]"
  else:
    flt_dur = ""
 
  if tag and type(tag) == str:
    flt_tag = "tag:"+tag
  else:
    flt_tag = ""

  # Querying Freesound
  page_size = 30
  if not flt_tag + flt_dur == "":
    qRes = fsClnt.text_search(query=queryText ,filter = flt_tag + flt_dur,sort="score", fields="id,name,previews,username,url,analysis", descriptors=','.join(descriptors), page_size=page_size, normalized=1)
  else:
    qRes = fsClnt.text_search(query=queryText ,sort="score",fields="id,name,previews,username,url,analysis", descriptors=','.join(descriptors), page_size=page_size, normalized=1)
  
  outDir2 = os.path.join(outputDir, queryText)
  if os.path.exists(outDir2):             # If the directory exists, it deletes it and starts fresh
      os.system("rm -r " + outDir2)
  os.mkdir(outDir2)

  pageNo = 1
  sndCnt = 0
  indCnt = 0
  totalSnds = min(qRes.count,200)   # System quits after trying to download after 200 times
  
  # Creating directories to store output and downloading sounds and their descriptors
  downloadedSounds = []
  while(1):
    if indCnt >= totalSnds:
      print("Not able to download required number of sounds. Either there are not enough search results on freesound for your search query and filtering constraints or something is wrong with this script.")
      break
    sound = qRes[indCnt - ((pageNo-1)*page_size)]
    print("Downloading mp3 preview and descriptors for sound with id: %s"%str(sound.id))
    outDir1 = os.path.join(outputDir, queryText, str(sound.id))
    if os.path.exists(outDir1):
      os.system("rm -r " + outDir1)
    os.system("mkdir " + outDir1)
    
    mp3Path = os.path.join(outDir1,  str(sound.previews.preview_lq_mp3.split("/")[-1]))
    ftrPath = mp3Path.replace('.mp3', featureExt)
    
    try:
      
      fs.FSRequest.retrieve(sound.previews.preview_hq_mp3, fsClnt, mp3Path)
      # Initialize a dictionary to store descriptors
      features = {}
      # Obtaining all the descriptors
      for desc in descriptors:
        features[desc]=[]
        features[desc].append(eval("sound.analysis."+desc))
      
      # Once we have all the descriptors, store them in a json file
      json.dump(features, open(ftrPath,'w'))
      sndCnt+=1
      downloadedSounds.append([str(sound.id), sound.url])

    except:
      if os.path.exists(outDir1):
        os.system("rm -r " + outDir1)
    
    indCnt +=1
    
    if indCnt%page_size==0:
      qRes = qRes.next_page()
      pageNo+=1
      
    if sndCnt>=topNResults:
      break
  # Dump the list of files and Freesound links
  fid = open(os.path.join(outDir2, queryText+'_SoundList.txt'), 'w')
  for elem in downloadedSounds:
    fid.write('\t'.join(elem)+'\n')
  fid.close()

In [ ]:
# E10-1.1: call download_sounds_freesound() for each of the chosen instrument classes
### your code here


**E10-1.2: Conceptual questions on sound collection (answer here)**

1. List the 10 (or more) instrument classes you chose and, for each, the `queryText`, `tag`, and `duration` settings you used. Explain why those settings yield coherent single-note/single-stroke samples.
2. Were any instrument classes harder to query than others? For one such class, describe what the initial results looked like and what you changed to improve coherence.
3. Why does the coherence of the 10-class dataset matter more for clustering than the 3-class dataset in E9? How does a larger number of classes amplify the effect of noisy or misclassified samples?
4. Which of your 10 instrument classes do you expect to be the most acoustically similar to each other (most likely to be confused), and why? Which pair do you expect to be easiest to separate?
5. Open one downloaded descriptor JSON file and inspect values for `sfx.logattacktime.mean` and `sfx.inharmonicity.mean` for one melodic and one percussive instrument. Do the values match your acoustic intuition?

## Part 2 – Baseline clustering performance

Establish a baseline k-means clustering accuracy for the 10-class dataset using Freesound descriptors, replicating the E9 methodology at larger scale.

- Visualise descriptor pairs to select a good subset for separation.
- Run `cluster_sounds()` with `nCluster = 10`.
- Report average accuracy over **10 runs** to account for random centroid initialisation.

**Reference:** Random chance for 10 classes is 10%. Your baseline should be higher but lower than what you achieved in E9 with 3 classes and careful selection.

In [ ]:
def convFtrDict2List(ftrDict):
  """
  This function converts descriptor dictionary to an np.array. The order in the numpy array (indices) 
  are same as those mentioned in descriptorMapping dictionary.
  
  Input: 
    ftrDict (dict): dictionary containing descriptors downloaded from the freesound
  Output: 
    ftr (np.ndarray): Numpy array containing the descriptors for processing later on
  """
  ftr = []
  for key in range(len(descriptorMapping.keys())):
    try:
      ftrName, ind = '.'.join(descriptorMapping[key].split('.')[:-1]), int(descriptorMapping[key].split('.')[-1])
      ftr.append(ftrDict[ftrName][0][ind])
    except:
      ftr.append(ftrDict[descriptorMapping[key]][0])
  return np.array(ftr)

def fetchDataDetails(inputDir, descExt = '.json'):
  """
  This function is used by other functions to obtain the information regarding the directory structure 
  and the location of descriptor files for each sound 
  """
  dataDetails = {}
  for path, dname, fnames  in os.walk(inputDir):
    for fname in fnames:
      if descExt in fname.lower():
        remain, rname, cname, sname = path.split('/')[:-3], path.split('/')[-3], path.split('/')[-2], path.split('/')[-1]
        if cname not in dataDetails:
          dataDetails[cname]={}
        fDict = json.load(open(os.path.join('/'.join(remain), rname, cname, sname, fname),'r'))
        dataDetails[cname][sname]={'file': fname, 'feature':fDict}
  return dataDetails

def cluster_sounds(targetDir, nCluster = -1, descInput=[]):
  """
  This function clusters all the sounds in targetDir using kmeans clustering.
  
  Input:
    targetDir (string): Directory where sound descriptors are stored (all the sounds in this 
                        directory will be used for clustering)
    nCluster (int): Number of clusters to be used for kmeans clustering.
    descInput (list) : List of indices of the descriptors to be used for similarity/distance 
                       computation (see descriptorMapping)
  Output:
    Prints the class of each cluster (computed by a majority vote), number of sounds in each 
    cluster and information (sound-id, sound-class and classification decision) of the sounds 
    in each cluster. Optionally, you can uncomment the return statement to return the same data.
  """
  
  dataDetails = fetchDataDetails(targetDir)
  
  ftrArr = []
  infoArr = []
  
  if nCluster ==-1:
    nCluster = len(dataDetails.keys())
  for cname in dataDetails.keys():
    #iterating over sounds
    for sname in dataDetails[cname].keys():
      ftrArr.append(convFtrDict2List(dataDetails[cname][sname]['feature'])[descInput])
      infoArr.append([sname, cname])
  
  ftrArr = np.array(ftrArr)
  infoArr = np.array(infoArr)
  
  ftrArrWhite = whiten(ftrArr)
  centroids, distortion = kmeans(ftrArrWhite, nCluster)
  clusResults = -1*np.ones(ftrArrWhite.shape[0])
  
  for ii in range(ftrArrWhite.shape[0]):
    diff = centroids - ftrArrWhite[ii,:]
    diff = np.sum(np.power(diff,2), axis = 1)
    indMin = np.argmin(diff)
    clusResults[ii] = indMin
  
  ClusterOut = []
  classCluster = []
  globalDecisions = []  
  for ii in range(nCluster):
    ind = np.where(clusResults==ii)[0]
    freqCnt = []
    for elem in infoArr[ind,1]:
      freqCnt.append(infoArr[ind,1].tolist().count(elem))
    indMax = np.argmax(freqCnt)
    classCluster.append(infoArr[ind,1][indMax])
    
    print("\n(Cluster: " + str(ii) + ") Using majority voting as a criterion this cluster belongs to " + 
          "class: " + classCluster[-1])
    print ("Number of sounds in this cluster are: " + str(len(ind)))
    decisions = []
    for jj in ind:
        if infoArr[jj,1] == classCluster[-1]:
            decisions.append(1)
        else:
            decisions.append(0)
    globalDecisions.extend(decisions)
    print ("sound-id, sound-class, classification decision")
    ClusterOut.append(np.hstack((infoArr[ind],np.array([decisions]).T)))
    print (ClusterOut[-1])
  globalDecisions = np.array(globalDecisions)
  totalSounds = len(globalDecisions)
  nIncorrectClassified = len(np.where(globalDecisions==0)[0])
  print("Out of %d sounds, %d sounds are incorrectly classified considering that one cluster should "
        "ideally contain sounds from only a single class"%(totalSounds, nIncorrectClassified))
  print("You obtain a classification (based on obtained clusters and majority voting) accuracy "
         "of %.2f percentage"%round(float(100.0*float(totalSounds-nIncorrectClassified)/totalSounds),2))
  # return ClusterOut

In [ ]:
# E10-2.1: run cluster_sounds() with Freesound descriptors to establish baseline accuracy
### your code here


**E10-2.2: Conceptual questions on baseline clustering (answer here)**

1. Report the descriptor subset you chose for the baseline and the average accuracy ± standard deviation over 10 runs. How does this compare with the E9 accuracy for 3 classes?
2. How much did accuracy vary across the 10 runs? Which cluster assignments were stable across runs and which were not? What does this tell you about how well your descriptors separate those classes?
3. Identify the two most frequently confused instrument pairs from your results. Examine their descriptor values — which descriptors are most similar between the two classes, and which are most different?
4. Why does the 10-class problem produce lower accuracy than the 3-class problem even with the same descriptor set? Give a geometric intuition in terms of cluster separation and feature space dimensionality.
5. Which descriptor(s) contributed most to what accuracy you did achieve? Attempt to remove one descriptor at a time and observe how accuracy changes — report the result for at least one descriptor removal.

## Part 3 – Improve clustering with Essentia

Improve on your Part 2 baseline by using Essentia to compute richer, cleaner descriptors. You must implement, at minimum, the following two improvements:

**Improvement 1 — More descriptors.**
Compute a broader set of Essentia descriptors (spectral, temporal, tonal, MFCCs, etc.) and select the subset that best separates your 10 instrument classes. A good starting point is the Essentia standard extractor (http://essentia.upf.edu/documentation/algorithms_overview.html#extractors), which outputs many frame-level low-level descriptors. Aggregate to sound-level statistics (mean, variance) as appropriate.

**Improvement 2 — Silent frame removal.**
Compute frame energy for each sound and plot it over time to choose a suitable energy threshold. Exclude silent frames (at the start and end of each recording) before computing mean descriptor statistics. A single global threshold typically works well for clean single-note recordings.

Beyond these two, you are free to explore additional improvements (normalisation, PCA, alternative distance measures, ensemble descriptors).

- Implement and call your Essentia descriptor extraction function.
- Call `cluster_sounds()` with new descriptors.

In [ ]:
# E10-3.1: implement Essentia descriptor extraction with silent-frame removal
### your code here


In [ ]:
# E10-3.2: call the Essentia feature extraction function for all sounds
### your code here


In [ ]:
# E10-3.3: run cluster_sounds() with Essentia descriptors and compare with baseline
### your code here


**E10-3.4: Conceptual questions on Part 3 improvements (answer here)**

1. List every Essentia descriptor (or descriptor group) you computed, with a one-sentence justification for each. Which ones were most discriminative for your 10-class problem, and how did you identify them?
2. Describe the silent-frame removal implementation. What energy threshold did you choose, and how did you validate it (e.g., plot the energy envelope for 2–3 sounds)? Did removing silent frames change descriptor values noticeably?
3. Report accuracy (average over 10 runs) for at least three descriptor subsets: (a) baseline Freesound descriptors only, (b) Essentia descriptors before silent-frame removal, and (c) Essentia descriptors after silent-frame removal. Summarise in a compact table.
4. Which instrument pairs remain most confused after your improvements? Explain the likely acoustic reason, and suggest one additional descriptor or pre-processing step that could separate them.
5. If you tried any further improvement beyond the two required ones (normalisation, PCA, alternative distance metric, etc.), describe what you did and the accuracy gain or loss you observed.